# P3 Community Detection and Camp Summaries

This notebook implements **Phase P3** from the frozen pipeline.

## Goals

- build the frozen row-normalized country projection for the capacity specification
- build the frozen row-normalized country projection for the count specification
- run Louvain community detection under the frozen seed and resolution
- export country-level community membership tables
- export community-level dominant mode profiles
- generate compact network summaries for the country projections
- create draft camp names and count higher-order merge candidates

## Required outputs

- `artifacts/tables/p3_capacity_community_membership.csv`
- `artifacts/tables/p3_count_community_membership.csv`
- `artifacts/tables/p3_capacity_dominant_mode_profiles.csv`
- `artifacts/tables/p3_count_dominant_mode_profiles.csv`
- `artifacts/tables/p3_count_higher_order_camp_map.csv`
- `artifacts/reports/p3_camp_summary_report.md`

Additional support outputs generated here:

- `artifacts/tables/p3_country_projection_network_summary.csv`
- `artifacts/runtime/p3_community_manifest.json`

This notebook uses the frozen `P2` matrices as its only analytical inputs.


In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p2_country_mode_count_raw.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P3__nogit'
SEED = 42
RESOLUTION = 1.0
TOP_K = 6
SIMILARITY = 'cosine'
EXPECTED_COUNTRIES = 94
EXPECTED_MODES = 16
EXPECTED_COUNT_TOTAL = 98942
EXPECTED_CAPACITY_TOTAL = 3573038.8
EXPECTED_TOP5 = {
    'count': 0.083786,
    'capacity': 0.092553,
}

manifest = {
    'notebook': 'P3_Community_Detection_and_Camp_Summaries.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'inputs': {
        'count_raw': str(TABLES / 'p2_country_mode_count_raw.csv'),
        'capacity_raw': str(TABLES / 'p2_country_mode_capacity_raw.csv'),
        'count_row_normalized': str(TABLES / 'p2_country_mode_count_row_normalized.csv'),
        'capacity_row_normalized': str(TABLES / 'p2_country_mode_capacity_row_normalized.csv'),
        'p2_diagnostics': str(TABLES / 'p2_concentration_diagnostics.csv'),
    },
    'frozen_rules': {
        'projection_similarity': SIMILARITY,
        'top_k': TOP_K,
        'seed': SEED,
        'resolution': RESOLUTION,
        'community_method': 'networkx.community.louvain_communities',
    },
}
(RUNTIME / 'p3_community_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Inputs:** `P2` raw + row-normalized matrices'))


## Load P2 Inputs and Verify Upstream Consistency

Before running community detection, this notebook verifies that the `P2` matrices still satisfy the frozen retained scope and the row-normalized projection diagnostics.


In [ ]:
count_raw = pd.read_csv(TABLES / 'p2_country_mode_count_raw.csv').set_index('Country/Area')
capacity_raw = pd.read_csv(TABLES / 'p2_country_mode_capacity_raw.csv').set_index('Country/Area')
count_row = pd.read_csv(TABLES / 'p2_country_mode_count_row_normalized.csv').set_index('Country/Area')
capacity_row = pd.read_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv').set_index('Country/Area')
p2_diag = pd.read_csv(TABLES / 'p2_concentration_diagnostics.csv')

mode_order = list(count_raw.columns)
country_order = list(count_raw.index)

for name, matrix in {
    'capacity_raw': capacity_raw,
    'count_row': count_row,
    'capacity_row': capacity_row,
}.items():
    if list(matrix.columns) != mode_order:
        raise ValueError(f'Mode order drift detected in {name}.')
    if list(matrix.index) != country_order:
        raise ValueError(f'Country order drift detected in {name}.')

for name, matrix in {
    'count_raw': count_raw,
    'capacity_raw': capacity_raw,
    'count_row': count_row,
    'capacity_row': capacity_row,
}.items():
    if matrix.shape != (EXPECTED_COUNTRIES, EXPECTED_MODES):
        raise ValueError(f'{name} shape drift: expected {(EXPECTED_COUNTRIES, EXPECTED_MODES)}, got {matrix.shape}')
    if matrix.isna().any().any():
        raise ValueError(f'{name} contains NaN values.')
    if (matrix < 0).any().any():
        raise ValueError(f'{name} contains negative values.')

if int(count_raw.to_numpy().sum()) != EXPECTED_COUNT_TOTAL:
    raise ValueError('count_raw total no longer matches frozen P1/P2 scope.')
if not math.isclose(float(capacity_raw.to_numpy().sum()), EXPECTED_CAPACITY_TOTAL, rel_tol=0, abs_tol=0.1):
    raise ValueError('capacity_raw total no longer matches frozen P1/P2 scope.')

count_row_sums = count_row.sum(axis=1).to_numpy(dtype=float)
capacity_row_sums = capacity_row.sum(axis=1).to_numpy(dtype=float)
if not np.isclose(count_row_sums, 1.0).all():
    raise ValueError('count_row is not properly row-normalized.')
if not np.isclose(capacity_row_sums, 1.0).all():
    raise ValueError('capacity_row is not properly row-normalized.')

input_check = pd.DataFrame({
    'matrix': ['count_raw', 'capacity_raw', 'count_row', 'capacity_row'],
    'shape': [count_raw.shape, capacity_raw.shape, count_row.shape, capacity_row.shape],
    'total_weight': [float(count_raw.to_numpy().sum()), float(capacity_raw.to_numpy().sum()), float(count_row.to_numpy().sum()), float(capacity_row.to_numpy().sum())],
    'row_sum_min': [float(count_raw.sum(axis=1).min()), float(capacity_raw.sum(axis=1).min()), float(count_row.sum(axis=1).min()), float(capacity_row.sum(axis=1).min())],
    'row_sum_max': [float(count_raw.sum(axis=1).max()), float(capacity_raw.sum(axis=1).max()), float(count_row.sum(axis=1).max()), float(capacity_row.sum(axis=1).max())],
})
display(input_check)


## Helper Functions

The helper functions below implement the frozen `P3` rules:

- row-normalized country projection
- cosine similarity with `top_k=6`
- Louvain community detection with `seed=42` and `resolution=1.0`
- stable relabeling by descending community size


In [ ]:
def split_mode(mode):
    return mode.split(' | ', 1)


def top_share(series, n=5):
    ordered = pd.Series(series, dtype=float).sort_values(ascending=False)
    total = float(ordered.sum())
    if total <= 0:
        return np.nan
    return float(ordered.head(n).sum() / total)


def build_projection_graph(matrix, similarity='cosine', top_k=6):
    names = list(matrix.index)
    values = matrix.to_numpy(dtype=float)
    graph = nx.Graph()
    graph.add_nodes_from(names)

    if len(names) < 2:
        return graph

    if similarity == 'cosine':
        sim = cosine_similarity(values)
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unknown similarity: {similarity}')

    np.fill_diagonal(sim, 0)
    k = min(top_k, len(names) - 1)

    for i, source in enumerate(names):
        neighbor_idx = np.argsort(sim[i])[-k:]
        for j in neighbor_idx:
            if i == j:
                continue
            weight = float(sim[i, j])
            if weight <= 0:
                continue
            target = names[j]
            if graph.has_edge(source, target):
                graph[source][target]['weight'] = max(graph[source][target]['weight'], weight)
            else:
                graph.add_edge(source, target, weight=weight)

    return graph


def detect_communities(graph, seed=42, resolution=1.0):
    if graph.number_of_nodes() == 0:
        return pd.Series(dtype=int), [], np.nan, pd.Series(dtype=float), pd.Series(dtype=float)

    if graph.number_of_edges() == 0:
        assignment = pd.Series({node: idx + 1 for idx, node in enumerate(sorted(graph.nodes()))}, name='community')
        communities = [{node} for node in assignment.index]
        strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
        degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
        return assignment, communities, 0.0, strengths, degrees

    communities = list(nx.community.louvain_communities(graph, weight='weight', seed=seed, resolution=resolution))
    communities = sorted(communities, key=len, reverse=True)
    assignment = pd.Series(
        {node: cid for cid, members in enumerate(communities, start=1) for node in members},
        name='community',
    ).sort_index()
    strengths = pd.Series(dict(graph.degree(weight='weight')), dtype=float).reindex(assignment.index).fillna(0)
    degrees = pd.Series(dict(graph.degree()), dtype=float).reindex(assignment.index).fillna(0)
    modularity = float(nx.community.modularity(graph, communities, weight='weight'))
    return assignment, communities, modularity, strengths, degrees


def draft_name_from_profile(profile):
    ranked = pd.Series(profile, dtype=float).sort_values(ascending=False)
    first_mode = ranked.index[0]
    second_mode = ranked.index[1]
    first_status, first_tier = split_mode(first_mode)
    second_status, _ = split_mode(second_mode)
    first_share = float(ranked.iloc[0])
    second_share = float(ranked.iloc[1])

    if second_status != first_status and second_share >= 0.15:
        return f'Draft: {first_status} / {second_status} mixed ({first_tier}-led)'
    if first_share >= 0.45:
        return f'Draft: {first_status} {first_tier} heavy'
    if first_share >= 0.30:
        return f'Draft: {first_status} {first_tier} led'
    return f'Draft: diversified {first_status} {first_tier}'


def summarize_country_projection(weight_kind, raw_matrix, row_matrix):
    graph = build_projection_graph(row_matrix, similarity=SIMILARITY, top_k=TOP_K)
    assignment, communities, modularity, strengths, degrees = detect_communities(graph, seed=SEED, resolution=RESOLUTION)

    if len(assignment) != len(row_matrix.index):
        raise ValueError(f'{weight_kind}: community assignment does not cover all countries.')
    if sorted(assignment.unique().tolist()) != list(range(1, assignment.nunique() + 1)):
        raise ValueError(f'{weight_kind}: community labels are not consecutive after relabeling.')

    observed_top5 = round(top_share(strengths, 5), 6)
    expected_top5 = EXPECTED_TOP5[weight_kind]
    if not math.isclose(observed_top5, expected_top5, rel_tol=0, abs_tol=1e-6):
        raise ValueError(f'{weight_kind}: projection top-5 share drifted from P2. Expected {expected_top5}, got {observed_top5}.')

    dominant_country_mode = row_matrix.idxmax(axis=1)
    dominant_country_mode_share = row_matrix.max(axis=1)
    community_sizes = assignment.value_counts().sort_index()
    profile_rows = []
    membership_rows = []

    for cid in sorted(assignment.unique()):
        members = assignment[assignment == cid].index.tolist()
        profile = row_matrix.loc[members].mean(axis=0).astype(float)
        ranked = profile.sort_values(ascending=False)
        top3 = ranked.head(3)
        name = draft_name_from_profile(profile)
        community_strength_sum = float(strengths.loc[members].sum())
        raw_total_weight = float(raw_matrix.loc[members].to_numpy().sum())

        profile_row = {
            'community': int(cid),
            'community_name_draft': name,
            'countries': int(len(members)),
            'country_share': float(len(members) / len(row_matrix)),
            'raw_total_weight': raw_total_weight,
            'raw_weight_share': float(raw_total_weight / raw_matrix.to_numpy().sum()),
            'graph_strength_sum': community_strength_sum,
            'graph_strength_share': float(community_strength_sum / strengths.sum()),
            'dominant_mode': top3.index[0],
            'dominant_mode_share': float(top3.iloc[0]),
            'second_mode': ranked.index[1],
            'second_mode_share': float(ranked.iloc[1]),
            'top3_modes': ' / '.join([f'{mode} ({share:.2f})' for mode, share in top3.items()]),
            'top3_share': float(top3.sum()),
            'dominant_status': split_mode(top3.index[0])[0],
            'dominant_tier': split_mode(top3.index[0])[1],
            'interpretable': bool(top3.iloc[0] >= 0.25 and top3.sum() >= 0.60),
            'members_sample': ', '.join(members[:8]),
        }
        for mode in row_matrix.columns:
            profile_row[f'profile__{mode}'] = float(profile[mode])
        profile_rows.append(profile_row)

        for country in members:
            membership_rows.append({
                'Country/Area': country,
                'community': int(cid),
                'community_name_draft': name,
                'graph_strength': float(strengths.loc[country]),
                'graph_degree': int(degrees.loc[country]),
                'raw_total_weight_country': float(raw_matrix.loc[country].sum()),
                'dominant_country_mode': dominant_country_mode.loc[country],
                'dominant_country_mode_share': float(dominant_country_mode_share.loc[country]),
                'community_dominant_mode': top3.index[0],
            })

    profile_df = pd.DataFrame(profile_rows).sort_values(['countries', 'raw_total_weight'], ascending=[False, False]).reset_index(drop=True)
    membership_df = pd.DataFrame(membership_rows).sort_values(['community', 'graph_strength'], ascending=[True, False]).reset_index(drop=True)
    graph_summary = pd.DataFrame([{
        'weight_kind': weight_kind,
        'similarity': SIMILARITY,
        'top_k': TOP_K,
        'nodes': int(graph.number_of_nodes()),
        'edges': int(graph.number_of_edges()),
        'density': float(nx.density(graph)),
        'communities': int(len(communities)),
        'modularity': modularity,
        'largest_community_size': int(community_sizes.max()),
        'largest_community_share': float(community_sizes.max() / len(assignment)),
        'top5_strength_share': observed_top5,
        'mean_graph_degree': float(degrees.mean()),
        'mean_graph_strength': float(strengths.mean()),
    }])

    return {
        'graph': graph,
        'assignment': assignment,
        'communities': communities,
        'strengths': strengths,
        'degrees': degrees,
        'membership': membership_df,
        'profiles': profile_df,
        'graph_summary': graph_summary,
    }


def build_count_higher_order_map(profile_df, target_groups=5):
    profile_columns = [col for col in profile_df.columns if col.startswith('profile__')]
    profile_lookup = {}
    for _, row in profile_df.iterrows():
        cid = int(row['community'])
        profile_lookup[cid] = row

    groups = {
        cid: {
            'members': [cid],
            'countries': int(profile_lookup[cid]['countries']),
            'profile': profile_lookup[cid][profile_columns].to_numpy(dtype=float),
        }
        for cid in profile_lookup
    }

    while len(groups) > target_groups:
        keys = sorted(groups)
        best_pair = None
        best_similarity = -1.0
        for i, left in enumerate(keys[:-1]):
            for right in keys[i + 1:]:
                similarity = float(cosine_similarity(
                    groups[left]['profile'].reshape(1, -1),
                    groups[right]['profile'].reshape(1, -1),
                )[0, 0])
                if similarity > best_similarity + 1e-12 or (
                    abs(similarity - best_similarity) <= 1e-12 and (best_pair is None or (left, right) < best_pair)
                ):
                    best_pair = (left, right)
                    best_similarity = similarity

        left, right = best_pair
        total_countries = groups[left]['countries'] + groups[right]['countries']
        merged_profile = (
            groups[left]['profile'] * groups[left]['countries'] + groups[right]['profile'] * groups[right]['countries']
        ) / total_countries
        groups[left] = {
            'members': sorted(groups[left]['members'] + groups[right]['members']),
            'countries': total_countries,
            'profile': merged_profile,
        }
        del groups[right]

    ordered_groups = sorted(groups.items(), key=lambda item: (-item[1]['countries'], item[0]))
    rows = []
    for gid, (anchor, info) in enumerate(ordered_groups, start=1):
        merged_profile = pd.Series(info['profile'], index=[col.replace('profile__', '') for col in profile_columns])
        merged_name = draft_name_from_profile(merged_profile)
        merged_dominant_mode = merged_profile.sort_values(ascending=False).index[0]
        for cid in info['members']:
            original = profile_lookup[cid]
            original_profile = original[profile_columns].to_numpy(dtype=float)
            rows.append({
                'original_count_community': int(cid),
                'original_community_name_draft': original['community_name_draft'],
                'original_dominant_mode': original['dominant_mode'],
                'higher_order_camp_candidate': int(gid),
                'higher_order_name_draft': merged_name,
                'group_members': ', '.join(str(member) for member in info['members']),
                'merge_recommended': bool(len(info['members']) > 1),
                'anchor_community': int(anchor),
                'profile_cosine_similarity_to_group': float(cosine_similarity(
                    original_profile.reshape(1, -1),
                    info['profile'].reshape(1, -1),
                )[0, 0]),
                'rationale': (
                    'Retain as standalone candidate.'
                    if len(info['members']) == 1
                    else f'Merged into candidate group because of high profile similarity around {merged_dominant_mode}.'
                ),
            })

    return pd.DataFrame(rows).sort_values('original_count_community').reset_index(drop=True)


## Run the Frozen Community Path

The capacity view is the main camp specification. The count view is the complementary ecology specification. Both are based on the row-normalized country projection.


In [ ]:
capacity_results = summarize_country_projection('capacity', capacity_raw, capacity_row)
count_results = summarize_country_projection('count', count_raw, count_row)
network_summary = pd.concat([
    capacity_results['graph_summary'],
    count_results['graph_summary'],
], ignore_index=True)
count_higher_order_map = build_count_higher_order_map(count_results['profiles'], target_groups=5)

capacity_results['membership'].to_csv(TABLES / 'p3_capacity_community_membership.csv', index=False)
count_results['membership'].to_csv(TABLES / 'p3_count_community_membership.csv', index=False)
capacity_results['profiles'].to_csv(TABLES / 'p3_capacity_dominant_mode_profiles.csv', index=False)
count_results['profiles'].to_csv(TABLES / 'p3_count_dominant_mode_profiles.csv', index=False)
count_higher_order_map.to_csv(TABLES / 'p3_count_higher_order_camp_map.csv', index=False)
network_summary.to_csv(TABLES / 'p3_country_projection_network_summary.csv', index=False)

if capacity_results['membership']['Country/Area'].nunique() != EXPECTED_COUNTRIES:
    raise ValueError('Capacity membership does not cover all retained countries.')
if count_results['membership']['Country/Area'].nunique() != EXPECTED_COUNTRIES:
    raise ValueError('Count membership does not cover all retained countries.')
if int(capacity_results['profiles']['countries'].sum()) != EXPECTED_COUNTRIES:
    raise ValueError('Capacity community sizes do not sum to the retained country count.')
if int(count_results['profiles']['countries'].sum()) != EXPECTED_COUNTRIES:
    raise ValueError('Count community sizes do not sum to the retained country count.')
if count_higher_order_map['original_count_community'].nunique() != count_results['profiles']['community'].nunique():
    raise ValueError('Count higher-order map does not cover all original count communities.')

display(Markdown('**Country projection network summary**'))
display(network_summary)
display(Markdown('**Capacity dominant mode profiles**'))
display(capacity_results['profiles'][['community', 'community_name_draft', 'countries', 'dominant_mode', 'dominant_mode_share', 'top3_modes']])
display(Markdown('**Count dominant mode profiles**'))
display(count_results['profiles'][['community', 'community_name_draft', 'countries', 'dominant_mode', 'dominant_mode_share', 'top3_modes']])
display(Markdown('**Count higher-order candidate map**'))
display(count_higher_order_map)


## Write the P3 Summary Report

This report preserves the main interpretive boundary of `P3`: the notebook identifies algorithmic communities and draft names, but it does not yet claim that compressed higher-order count camps are final. Those remain candidates pending later audit.


In [ ]:
report_lines = [
    '# P3 Camp Summary Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    '- inputs: frozen `P2` raw and row-normalized country-by-mode matrices',
    '',
    '## Frozen rules used',
    f'- projection: row-based country projection using `{SIMILARITY}` similarity and `top_k={TOP_K}`',
    f'- community detection: `networkx.community.louvain_communities` with `seed={SEED}` and `resolution={RESOLUTION}`',
    '- interpretation boundary: preserve original algorithmic communities in the analysis outputs',
    '',
    '## Input QA',
    f'- matrices checked: `(94, 16)` for count raw, capacity raw, count row-normalized, and capacity row-normalized',
    f'- raw totals checked: `98,942` count and `3,573,038.8` MW',
    '- row-normalized matrices checked: all country rows sum to 1.0, with no NaN and no negative entries',
    '',
    '## Country projection network summary',
]
for row in network_summary.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: nodes={row.nodes}, edges={row.edges}, density={row.density:.6f}, communities={row.communities}, modularity={row.modularity:.6f}, top5_strength_share={row.top5_strength_share:.6f}'
    )

report_lines.extend([
    '',
    '## Capacity camp draft summary',
])
for row in capacity_results['profiles'].itertuples(index=False):
    report_lines.append(
        f'- community {row.community}: {row.community_name_draft}; countries={row.countries}; dominant_mode={row.dominant_mode}; top3={row.top3_modes}'
    )

report_lines.extend([
    '',
    '## Count camp draft summary',
])
for row in count_results['profiles'].itertuples(index=False):
    report_lines.append(
        f'- community {row.community}: {row.community_name_draft}; countries={row.countries}; dominant_mode={row.dominant_mode}; top3={row.top3_modes}'
    )

report_lines.extend([
    '',
    '## Count higher-order candidate map',
    '- This map is provisional and does not replace the original count communities.',
])
for row in count_higher_order_map.itertuples(index=False):
    report_lines.append(
        f'- count community {row.original_count_community} -> candidate {row.higher_order_camp_candidate}: {row.higher_order_name_draft}; merge_recommended={bool(row.merge_recommended)}; rationale={row.rationale}'
    )

report_lines.extend([
    '',
    '## Interpretation boundary after P3',
    '- Capacity communities are the main camp structure for substantive interpretation.',
    '- Count communities remain a complementary ecology view.',
    '- Any higher-order compression of count communities is still a draft candidate and must not overwrite the original algorithmic output at this stage.',
    '',
    '## Output inventory',
    '- `artifacts/tables/p3_capacity_community_membership.csv`',
    '- `artifacts/tables/p3_count_community_membership.csv`',
    '- `artifacts/tables/p3_capacity_dominant_mode_profiles.csv`',
    '- `artifacts/tables/p3_count_dominant_mode_profiles.csv`',
    '- `artifacts/tables/p3_count_higher_order_camp_map.csv`',
    '- `artifacts/tables/p3_country_projection_network_summary.csv`',
    '- `artifacts/reports/p3_camp_summary_report.md`',
    '- `artifacts/runtime/p3_community_manifest.json`',
])

(REPORTS / 'p3_camp_summary_report.md').write_text('\n'.join(report_lines), encoding='utf-8')

display(Markdown((REPORTS / 'p3_camp_summary_report.md').read_text(encoding='utf-8')))
